# MALT annotation for E14S / E15S (MC38 spatial)

## Reference choice

We use **GEO [GSE193342](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE193342)** (Liu *et al.*, mouse colonic mucosa in DSS colitis; PMID [37652012](https://pubmed.ncbi.nlm.nih.gov/37652012/)) as the scRNA reference. It is **mouse**, **10x-like UMI counts**, and contains a realistic **tumor-adjacent microenvironment** mix (strong **Immune**, **Mesenchyme**, **Epithelium**) that overlaps the cell types you expect around MC38 tumors better than a generic PBMC atlas.

Labels are **`coarse_annotation` + `_` + `annotation.2`** (e.g. `Immune_Macrophage`) so types stay interpretable when the same fine name appears under different lineages.

## Pipeline

1. `prepare_mouse_colon_reference_gse193342.py` — download GEO supplementary matrix + metadata, build `references/mouse_colon_gse193342_reference.h5ad` with `obs['cell_type']`.
2. `export_mc38_query_for_malt.py` — merge `E14S/` and `E15S/` `filtered_feature_bc_matrix.h5` (gene expression only) into `query_mc38_e14s_e15s_gex.h5ad`.
3. `marker_label_transfer.py` — MALT label transfer; use **`--labels-only-output`** to attach `malt_label` / `malt_confidence` / `knn_label` to the original query without bloating the file with duplicate dense matrices.

**Note:** If PyTorch was built against NumPy 1.x but your environment has NumPy 2.x, this script converts tensors via `.tolist()` so MALT still runs (you may see a one-time warning).

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path(".").resolve()
print("ROOT", ROOT)

### 1. Build reference (~334 MB download + ~20 GB RAM peak for full matrix load)

Runs once; files are cached under `references/gse193342_cache/`.

In [ ]:
subprocess.check_call(
    [sys.executable, str(ROOT / "prepare_mouse_colon_reference_gse193342.py")],
    cwd=ROOT,
)

### 2. Export merged MC38 query (gene expression only)

In [ ]:
subprocess.check_call(
    [sys.executable, str(ROOT / "export_mc38_query_for_malt.py")],
    cwd=ROOT,
)

### 3. Run MALT

Adjust `--extra-markers` with genes you care about on the tumor section.

In [ ]:
cmd = [
    sys.executable,
    str(ROOT / "marker_label_transfer.py"),
    "--reference",
    str(ROOT / "references" / "mouse_colon_gse193342_reference.h5ad"),
    "--query",
    str(ROOT / "query_mc38_e14s_e15s_gex.h5ad"),
    "--groupby",
    "cell_type",
    "--outdir",
    str(ROOT / "malt_out"),
    "--output-query",
    "query_malt_labeled.h5ad",
    "--labels-only-output",
    "--extra-markers",
    "Cd274,Pdcd1lg2,Cd3e,Cd79a,Lyz2,S100a8",
]
print(" ".join(cmd))
subprocess.check_call(cmd, cwd=ROOT)

### 4. Inspect results in Python

In [ ]:
import scanpy as sc

q = sc.read_h5ad(ROOT / "malt_out" / "query_malt_labeled.h5ad")
print(q.obs["malt_label"].value_counts().head(15))
q.obs[["sample", "malt_label", "malt_confidence"]].head()